## SOTA_MATH unified evaluation (Colab/Kaggle)

This notebook evaluates `Aniket200325/SOTA_MATH-phase4` in **pure reasoning mode** (no tools) on **MATH-Hard** from the Hugging Face repo `lighteval/MATH-Hard`.

It will:
- Download the per-subject `*.jsonl` files
- Combine them into one file: `artifacts/math_hard_combined.jsonl`
- Upload that combined set to **LangSmith** (so you can inspect failures)
- Run a LangSmith `evaluate()` experiment with an OpenAI-backed judge

Outputs:
- `artifacts/math_hard_combined.jsonl`
- (Optional) `artifacts/summary.csv` (if the LangSmith result object supports dataframe export)


In [1]:
# --- Colab/Kaggle setup ---
import os, sys, platform, textwrap, json, time, math, random
from pathlib import Path

# Set vLLM multiprocessing/env knobs BEFORE any torch/CUDA heavy init.
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
# T4/Colab fallback if v1 engine is unstable; set to "1" to force v1.
os.environ.setdefault("VLLM_USE_V1", "0")

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("CWD:", os.getcwd())
print("VLLM_WORKER_MULTIPROC_METHOD:", os.environ.get("VLLM_WORKER_MULTIPROC_METHOD"))
print("VLLM_USE_V1:", os.environ.get("VLLM_USE_V1"))


Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
CWD: /content


In [2]:
# Install deps (Colab/Kaggle) with Colab-safe pins
# These pins avoid breaking preinstalled Colab packages (pandas/opentelemetry/ipython stack).

%pip -q install \
  "openai>=1.0.0,<2.0.0" \
  "datasets>=2.18.0,<4.0.0" \
  "huggingface_hub>=0.24.0,<1.0.0" \
  "sympy>=1.12,<2.0.0" \
  "pandas==2.2.2" \
  "tqdm>=4.66.0,<5.0.0" \
  "vllm>=0.5.0" \
  "langsmith>=0.1.0" \
  "langchain-core>=0.2.0" \
  "langchain-openai>=0.2.0" \
  "opentelemetry-api>=1.36.0,<1.39.0" \
  "opentelemetry-sdk>=1.36.0,<1.39.0" \
  "jedi>=0.19.0,<0.20.0"

print("If this cell changed core packages, restart runtime once before continuing.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 153.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 5.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 53.2 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 129.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [ ]:
from google.colab import userdata
import os

os.environ["LANGCHAIN_API_KEY"] = "LANGCHAIN_API_KEY"
os.environ["OPENAI_API_KEY"] = "OPENAI_API_KEY"
os.environ["HF_TOKEN"] = "HF_TOKEN"

In [ ]:
# GPU sanity check
try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
        print("capability:", torch.cuda.get_device_capability(0))
        props = torch.cuda.get_device_properties(0)
        print("vram_gb:", round(props.total_memory / (1024**3), 2))
except Exception as e:
    print("GPU check failed:", e)


torch: 2.10.0+cu128
cuda available: True
gpu: NVIDIA A100-SXM4-80GB
capability: (8, 0)
vram_gb: 79.25


### Config

Tune these based on your GPU:
- `DTYPE`: use `"bfloat16"` on A100/H100/L4; use `"float16"` on T4.
- `GPU_MEMORY_UTIL`: lower if you OOM.
- `MAX_MODEL_LEN`: keep at 4096 (matches your agent prompt/template).


In [ ]:
MODEL_PATH = os.environ.get("MODEL_PATH", "Aniket200325/SOTA_MATH-phase4")
DTYPE = os.environ.get("DTYPE", "bfloat16")  # set to float16 for T4
TENSOR_PARALLEL_SIZE = int(os.environ.get("TENSOR_PARALLEL_SIZE", "1"))
MAX_MODEL_LEN = int(os.environ.get("MAX_MODEL_LEN", "4096"))
GPU_MEMORY_UTIL = float(os.environ.get("GPU_MEMORY_UTIL", "0.90"))

# Agent loop
MAX_ITERATIONS = int(os.environ.get("MAX_ITERATIONS", "20"))
GEN_TEMPERATURE = float(os.environ.get("GEN_TEMPERATURE", "0.0"))
GEN_MAX_TOKENS = int(os.environ.get("GEN_MAX_TOKENS", "3072"))

# MATH-Hard
MATH_HARD_REPO = os.environ.get("MATH_HARD_REPO", "lighteval/MATH-Hard")
MATH_HARD_FILES = [
    "algebra.jsonl",
    "counting_and_probability.jsonl",
    "geometry.jsonl",
    "intermediate_algebra.jsonl",
    "number_theory.jsonl",
    "prealgebra.jsonl",
    "precalculus.jsonl",
]
SEED = int(os.environ.get("SEED", "42"))
# Optional cap for quick runs (0 = use full dataset)
MAX_EXAMPLES = int(os.environ.get("MAX_EXAMPLES", "0"))

# LangSmith
LANGSMITH_DATASET_NAME = os.environ.get("LANGSMITH_DATASET_NAME", "MATH_Hard_All")
LANGSMITH_EXPERIMENT_PREFIX = os.environ.get("LANGSMITH_EXPERIMENT_PREFIX", "sota-math-math-hard")
MAX_CONCURRENCY = int(os.environ.get("MAX_CONCURRENCY", "1"))

print({
    "MODEL_PATH": MODEL_PATH,
    "DTYPE": DTYPE,
    "MAX_MODEL_LEN": MAX_MODEL_LEN,
    "GPU_MEMORY_UTIL": GPU_MEMORY_UTIL,
    "MAX_ITERATIONS": MAX_ITERATIONS,
    "GEN_MAX_TOKENS": GEN_MAX_TOKENS,
    "MATH_HARD_REPO": MATH_HARD_REPO,
    "MAX_EXAMPLES": MAX_EXAMPLES,
    "LANGSMITH_DATASET_NAME": LANGSMITH_DATASET_NAME,
    "LANGSMITH_EXPERIMENT_PREFIX": LANGSMITH_EXPERIMENT_PREFIX,
    "MAX_CONCURRENCY": MAX_CONCURRENCY,
})


{'MODEL_PATH': 'Aniket200325/SOTA_MATH-phase4', 'DTYPE': 'bfloat16', 'MAX_MODEL_LEN': 4096, 'GPU_MEMORY_UTIL': 0.9, 'MAX_ITERATIONS': 20, 'GEN_MAX_TOKENS': 2048, 'MATH_HARD_REPO': 'lighteval/MATH-Hard', 'MAX_EXAMPLES': 0, 'LANGSMITH_DATASET_NAME': 'MATH_Hard_All', 'LANGSMITH_EXPERIMENT_PREFIX': 'sota-math-math-hard', 'MAX_CONCURRENCY': 1}


In [ ]:
# Optional: T4-safe preset for smoke tests
# Run this cell only when you're on T4 and want a stable quick run.
USE_T4_PRESET = os.environ.get("USE_T4_PRESET", "0") == "1"

if USE_T4_PRESET:
    DTYPE = "float16"
    MAX_MODEL_LEN = 2048
    GEN_MAX_TOKENS = 1024
    GPU_MEMORY_UTIL = 0.75
    MAX_EXAMPLES = min(MAX_EXAMPLES if MAX_EXAMPLES > 0 else 10, 10)
    MAX_CONCURRENCY = 1
    print("Applied T4 preset:")
    print({
        "DTYPE": DTYPE,
        "MAX_MODEL_LEN": MAX_MODEL_LEN,
        "GEN_MAX_TOKENS": GEN_MAX_TOKENS,
        "GPU_MEMORY_UTIL": GPU_MEMORY_UTIL,
        "MAX_EXAMPLES": MAX_EXAMPLES,
        "MAX_CONCURRENCY": MAX_CONCURRENCY,
    })
else:
    print("T4 preset not enabled. Set USE_T4_PRESET=1 to enable.")


In [ ]:
# No tool helpers needed: evaluation runs in pure reasoning mode.


In [ ]:
# --- vLLM agent (reasoning-only, no tools) ---
from vllm import LLM, SamplingParams

SYSTEM_PROMPT = """You are an expert mathematical AI assistant.

Solve the problem carefully and provide concise but complete reasoning.
Finish with a final line in this exact format:
**Answer:** <final answer>

Do not use any external tools. Complete the full solution directly.
"""


def build_prompt(messages: list[dict]) -> str:
    prompt = ""
    for msg in messages:
        prompt += f"<|start_header_id|>{msg['role']}<|end_header_id|>\n\n{msg['content']}<|eot_id|>"
    prompt += "<|start_header_id|>assistant<|end_header_id|>\n\n"
    return prompt


def extract_final_answer(text: str) -> str:
    ans_idx = text.rfind("**Answer:**")
    if ans_idx != -1:
        return text[ans_idx + len("**Answer:**") :].strip()
    ans_idx = text.rfind("Answer:")
    if ans_idx != -1:
        return text[ans_idx + len("Answer:") :].strip()
    return text.strip()


llm = None
sampling_params = None

def load_llm():
    global llm, sampling_params
    if llm is not None:
        return llm

    print("Loading model via vLLM…")
    llm = LLM(
        model=MODEL_PATH,
        dtype=DTYPE,
        tensor_parallel_size=TENSOR_PARALLEL_SIZE,
        max_model_len=MAX_MODEL_LEN,
        gpu_memory_utilization=GPU_MEMORY_UTIL,
        trust_remote_code=True,
    )
    sampling_params = SamplingParams(
        temperature=GEN_TEMPERATURE,
        max_tokens=GEN_MAX_TOKENS,
        stop=["<|eot_id|>"],
    )
    return llm


def run_agent(question: str, max_iterations: int = MAX_ITERATIONS) -> dict:
    load_llm()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    t0 = time.time()
    prompt = build_prompt(messages)
    out = llm.generate([prompt], sampling_params, use_tqdm=False)
    generated_text = out[0].outputs[0].text
    messages.append({"role": "assistant", "content": generated_text})

    elapsed_s = time.time() - t0
    full_trace = generated_text
    pred = extract_final_answer(full_trace)

    return {
        "prediction": pred,
        "full_reasoning_trace": full_trace,
        "history": messages,
        "elapsed_s": elapsed_s,
    }


In [ ]:
# --- Download + combine MATH-Hard into one JSONL ---
from huggingface_hub import hf_hub_download
from datasets import load_dataset
import ast

COMBINED_JSONL = ARTIFACT_DIR / "math_hard_combined.jsonl"


def _parse_record(text: str):
    text = text.strip().lstrip("\ufeff")
    if not text:
        return None

    # Strict JSON first
    try:
        return json.loads(text)
    except Exception:
        pass

    # Python-literal fallback (handles single quotes, True/False/None)
    try:
        return ast.literal_eval(text)
    except Exception:
        return None


def _iter_records(path: str):
    # Supports JSONL and JSON arrays; skips malformed lines.
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read().lstrip("\ufeff").strip()

    if not raw:
        return

    # If file is a full JSON array, parse once.
    if raw.startswith("["):
        arr = _parse_record(raw)
        if isinstance(arr, list):
            for obj in arr:
                if isinstance(obj, dict):
                    yield obj
            return

    # Otherwise parse line-by-line.
    bad = 0
    for i, line in enumerate(raw.splitlines(), start=1):
        line = line.strip()
        if not line:
            continue
        obj = _parse_record(line)
        if isinstance(obj, dict):
            yield obj
        else:
            bad += 1
            if bad <= 3:
                print(f"Warning: skipped malformed record in {path} at line {i}: {line[:120]}")


def _download_math_hard_file(repo_id: str, fname: str) -> str:
    token = os.environ.get("HF_TOKEN")
    # Files are in repo subfolder: test/
    try:
        return hf_hub_download(repo_id=repo_id, filename=f"test/{fname}", repo_type="dataset", token=token)
    except Exception:
        # Fallback in case layout changes
        return hf_hub_download(repo_id=repo_id, filename=fname, repo_type="dataset", token=token)


def download_and_combine_math_hard(repo_id: str = MATH_HARD_REPO, files: list[str] = MATH_HARD_FILES) -> Path:
    out_lines = 0
    with COMBINED_JSONL.open("w", encoding="utf-8") as out:
        for fname in files:
            local_path = _download_math_hard_file(repo_id=repo_id, fname=fname)
            file_rows = 0
            for obj in _iter_records(local_path):
                obj = dict(obj)
                obj.setdefault("subject", fname.replace(".jsonl", ""))
                out.write(json.dumps(obj, ensure_ascii=False) + "\n")
                out_lines += 1
                file_rows += 1
            print(f"{fname}: {file_rows} rows")

    print("Wrote combined JSONL:", str(COMBINED_JSONL))
    print("Total rows:", out_lines)
    return COMBINED_JSONL


download_and_combine_math_hard()

# Load combined as a HF dataset (so we can sample/shuffle/cap)
ds = load_dataset("json", data_files=str(COMBINED_JSONL), split="train")

ds = ds.shuffle(seed=SEED)
if MAX_EXAMPLES and MAX_EXAMPLES > 0:
    ds = ds.select(range(min(MAX_EXAMPLES, len(ds))))

print("Loaded rows:", len(ds))
print("Columns:", ds.column_names)
print("Example keys:", list(ds[0].keys()))


algebra.jsonl: 0.00B [00:00, ?B/s]

JSONDecodeError: Expecting value: line 1 column 2 (char 1)

: 

: 

In [ ]:
# --- LangSmith setup + judge (OpenAI-backed via LangChain) ---
# Required:
# - LANGCHAIN_API_KEY (LangSmith)
# - OPENAI_API_KEY (judge model)

from langsmith import Client
from langsmith import evaluate
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "gpt-4o-mini")

# Helps LangSmith auto-link traces (optional but recommended)
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGCHAIN_PROJECT", LANGSMITH_EXPERIMENT_PREFIX)

ls_client = Client()

# Force structured JSON output when supported by the runtime.
# (Some older model/SDK combos may ignore this; we still parse robustly below.)
judge_llm = ChatOpenAI(
    model=JUDGE_MODEL,
    temperature=0.0,
    model_kwargs={"response_format": {"type": "json_object"}},
)

judge_chain = (
    PromptTemplate.from_template(
        "You are an expert math grader.\n\n"
        "Return ONLY a JSON object (no markdown, no extra text) with keys:\n"
        "- verdict: 'YES' or 'NO'\n"
        "- reason: a short reason (<= 20 words)\n\n"
        "Question: {question}\n\n"
        "Ground Truth Solution/Answer: {ground_truth}\n\n"
        "Student's Predicted Final Answer: {prediction}\n"
    )
    | judge_llm
    | StrOutputParser()
)

# Best-effort token accounting (LangChain/OpenAI usage metadata varies by version)
openai_usage_acc = {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}


def _parse_judge_json(text: str) -> dict:
    """Parse strict JSON; also tolerates accidental prefix/suffix."""
    t = (text or "").strip()
    # Fast path
    try:
        obj = json.loads(t)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # Try to extract the first {...} block
    m = re.search(r"\{[\s\S]*\}", t)
    if m:
        try:
            obj = json.loads(m.group(0))
            if isinstance(obj, dict):
                return obj
        except Exception:
            pass

    return {"verdict": "NO", "reason": f"Unparseable judge output: {t[:200]}"}


def correctness_evaluator(run, example) -> dict:
    question = example.inputs.get("question", "")
    ground_truth = example.outputs.get("ground_truth", "")
    prediction = (run.outputs or {}).get("prediction", "")

    try:
        raw = judge_chain.invoke({
            "question": question,
            "ground_truth": ground_truth,
            "prediction": prediction,
        })
        obj = _parse_judge_json(raw)

        verdict = str(obj.get("verdict", "NO")).strip().upper()
        if verdict not in {"YES", "NO"}:
            verdict = "NO"
        reason = str(obj.get("reason", "")).strip()
        score = 1 if verdict == "YES" else 0

        # Store JSON in comment so it's easy to review/export
        return {"key": "correctness", "score": score, "comment": json.dumps({"verdict": verdict, "reason": reason}, ensure_ascii=False)}
    except Exception as e:
        return {"key": "correctness", "score": 0, "comment": json.dumps({"verdict": "NO", "reason": f"Judge failed: {e}"})}



In [ ]:
# --- Upload MATH-Hard to LangSmith, then run evaluation ---
import pandas as pd

SUMMARY_PATH = ARTIFACT_DIR / "summary.csv"


def _normalize_math_hard_row(row: dict) -> dict:
    # MATH-Hard JSONL schema can vary; normalize to what we need.
    question = row.get("problem") or row.get("question") or row.get("prompt") or ""
    ground_truth = row.get("solution") or row.get("answer") or row.get("ground_truth") or ""
    subject = row.get("subject") or row.get("type") or row.get("category") or "unknown"
    level = row.get("level")
    return {
        "question": str(question),
        "ground_truth": str(ground_truth),
        "subject": str(subject),
        "level": None if level is None else str(level),
    }


def ensure_langsmith_dataset_from_ds(ds, dataset_name: str = LANGSMITH_DATASET_NAME, recreate: bool = True):
    # Requires LANGCHAIN_API_KEY
    existing = list(ls_client.list_datasets(dataset_name=dataset_name))
    if existing and recreate:
        ls_client.delete_dataset(dataset_id=existing[0].id)
        existing = []

    if existing:
        dataset = existing[0]
    else:
        dataset = ls_client.create_dataset(
            dataset_name=dataset_name,
            description=f"Combined MATH-Hard JSONL from {MATH_HARD_REPO}",
        )

    inputs = []
    outputs = []
    metadatas = []
    for r in ds:
        ex = _normalize_math_hard_row(r)
        inputs.append({"question": ex["question"]})
        outputs.append({"ground_truth": ex["ground_truth"]})
        metadatas.append({"subject": ex["subject"], "level": ex["level"]})

    ls_client.create_examples(inputs=inputs, outputs=outputs, metadata=metadatas, dataset_id=dataset.id)
    print("LangSmith dataset ready:", dataset_name, "examples:", len(inputs))
    return dataset_name


def math_agent_target(inputs: dict) -> dict:
    out = run_agent(inputs["question"], max_iterations=MAX_ITERATIONS)
    return {
        "prediction": out["prediction"],
        "full_reasoning_trace": out["full_reasoning_trace"],
        "elapsed_s": out["elapsed_s"],
    }


DATASET_NAME = ensure_langsmith_dataset_from_ds(ds, dataset_name=LANGSMITH_DATASET_NAME, recreate=True)

# Preflight: fail fast on backend startup issues once (instead of per-example retries).
print("Running vLLM preflight load...")
_ = load_llm()
print("vLLM preflight OK.")

experiment_results = evaluate(
    math_agent_target,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator],
    experiment_prefix=LANGSMITH_EXPERIMENT_PREFIX,
    max_concurrency=MAX_CONCURRENCY,
)

print("Evaluation launched/completed. Check LangSmith for traces and per-example failures.")

# Optional: attempt to materialize a small local summary if supported by this langsmith version
try:
    df = experiment_results.to_pandas()
    cols = [c for c in df.columns if "correct" in c.lower() or "elapsed" in c.lower()]
    display(df[cols].head())

    # A minimal aggregate across all examples
    summary = pd.DataFrame({
        "n": [len(df)],
        "avg_correctness": [df[[c for c in df.columns if "correctness" in c]].mean(numeric_only=True).iloc[0] if any("correctness" in c for c in df.columns) else None],
    })
    summary.to_csv(SUMMARY_PATH, index=False)
    print("Wrote:", SUMMARY_PATH)
    display(summary)
except Exception as e:
    print("Local summary export skipped:", e)


### Runtime + judge cost estimator

- **Agent runtime** depends heavily on average generated tokens/problem and tool-loop length.
- **Judge cost** still comes from OpenAI usage (because the judge model is OpenAI), even though the orchestration + trace storage is handled by LangSmith.

Use the cell below for rough time estimates. For judge cost, the most reliable source is the OpenAI usage dashboard; token usage attribution inside LangSmith/LC varies by SDK/runtime.


In [ ]:
# --- Rough wall-clock time estimates by GPU ---
# Ballpark decode throughputs for an 8B-class model in vLLM, single-GPU, batch=1.
# Your actual numbers can vary 2-3x depending on runtime, driver, context length, and paging.

GPU_TOK_S = {
    "T4_16GB": 60,
    "L4_24GB": 120,
    "RTX_R6000_48GB": 180,
    "A100_40GB": 250,
    "H100_80GB": 400,
}

N = len(ds)
ASSUMED_GEN_TOKENS_PER_PROBLEM = int(os.environ.get("ASSUMED_GEN_TOKENS_PER_PROBLEM", "700"))

print("N examples:", N)
print("Assumed generation tokens/problem:", ASSUMED_GEN_TOKENS_PER_PROBLEM)

print("\nEstimated agent-only total time (decode-only heuristic):")
for gpu, tps in GPU_TOK_S.items():
    sec = (N * ASSUMED_GEN_TOKENS_PER_PROBLEM) / tps
    print(f"{gpu}: ~{sec/3600:.2f} hours")


# --- OpenAI judge cost ---
# LangSmith orchestrates the judge, but the judge still bills against your OpenAI account.
# Token usage may or may not be exposed in your current LangChain/OpenAI versions.
# If you can capture token usage, set openai_usage_acc; otherwise estimate using assumed tokens.

INPUT_PER_1M_USD = float(os.environ.get("JUDGE_INPUT_USD_PER_1M", "0"))
OUTPUT_PER_1M_USD = float(os.environ.get("JUDGE_OUTPUT_USD_PER_1M", "0"))

ASSUMED_JUDGE_INPUT_TOKENS_PER_EX = int(os.environ.get("ASSUMED_JUDGE_INPUT_TOKENS_PER_EX", "900"))
ASSUMED_JUDGE_OUTPUT_TOKENS_PER_EX = int(os.environ.get("ASSUMED_JUDGE_OUTPUT_TOKENS_PER_EX", "40"))

it_obs = int(openai_usage_acc.get("input_tokens", 0) or 0)
ot_obs = int(openai_usage_acc.get("output_tokens", 0) or 0)

if it_obs == 0 and ot_obs == 0:
    it = N * ASSUMED_JUDGE_INPUT_TOKENS_PER_EX
    ot = N * ASSUMED_JUDGE_OUTPUT_TOKENS_PER_EX
    mode = "estimated"
else:
    it, ot = it_obs, ot_obs
    mode = "observed"

estimated_cost = (it / 1_000_000) * INPUT_PER_1M_USD + (ot / 1_000_000) * OUTPUT_PER_1M_USD

print("\nJudge model:", JUDGE_MODEL)
print("Judge tokens (mode=%s):" % mode, {"input_tokens": it, "output_tokens": ot, "total_tokens": it + ot})
print("Set env JUDGE_INPUT_USD_PER_1M and JUDGE_OUTPUT_USD_PER_1M to compute $.")
print("Estimated judge cost ($):", estimated_cost)
